# HMDB51 — распознавание действий: обучение в Colab (GPU)

Обучает **5 архитектур** (TSN, TSM, I3D, R(2+1)D, VideoMAE) на 10-классовом
подмножестве HMDB51. Данные — с **HuggingFace-зеркала** `jili5044/hmdb51`
(клипы уже .avi по папкам, RAR/unrar не нужны). Чекпоинты сохраняются на
**Google Drive каждую эпоху** (Colab отваливается по таймауту).

**Перед запуском:** Runtime → Change runtime type → **GPU**.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
!nvidia-smi -L

## 1. Google Drive (чтобы чекпоинты переживали перезапуск)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Получить проект
Укажите URL вашего GitHub-репозитория (после git init / push).

In [ ]:
import os
REPO_URL = "https://github.com/USER/venya.git"  # <-- ЗАМЕНИТЕ
if not os.path.exists("/content/venya"):
    !git clone $REPO_URL /content/venya
%cd /content/venya

## 3. Зависимости
Colab уже содержит CUDA-torch; ставим только недостающее.

In [ ]:
!pip -q install -r requirements-colab.txt
# unrar нужен ТОЛЬКО для запасного источника serre-lab:
!apt-get -qq install -y unrar > /dev/null 2>&1 || true

## 4. Данные с HuggingFace-зеркала
Тянет только наши 10 классов и раскладывает в data/raw/<class>/*.avi.

In [ ]:
!python -m src.data.download_hmdb --source hf
!python -m src.data.download_hmdb --check

## 5. Sanity-проверка на РЕАЛЬНЫХ .avi
Закрывает то единственное, что дал бы локальный прогон: реальный PyAV-декодер,
аугментации и сходимость сплитов.

In [ ]:
from src.data.splits import build_splits
from src.data.dataset import decode_video, sample_indices, transform_clip

sp = build_splits()
print("split source:", sp["_source"])
for k in ("train", "val", "test"):
    print(f"  {k:5s}: {len(sp[k])} clips")

for path, label in sp["test"][:3]:
    frames = decode_video(path)
    idx = sample_indices(len(frames), 8, train=False)
    clip = transform_clip(frames[idx], 112, train=True)
    print(os.path.basename(path), "| decoded", len(frames), "-> clip", tuple(clip.shape))
print("PyAV decode + augment + splits: OK")

## 6. Чекпоинты и результаты → на Drive

In [ ]:
import pathlib, shutil
DRIVE = "/content/drive/MyDrive/venya"
for name in ("checkpoints", "results"):
    os.makedirs(f"{DRIVE}/{name}", exist_ok=True)
    p = pathlib.Path(name)
    if p.is_symlink():
        p.unlink()
    elif p.exists():
        shutil.rmtree(p)
    os.symlink(f"{DRIVE}/{name}", name)
print("checkpoints/ и results/ -> Drive")

## 7. ФАЗА 1 — мини-прогон на реальных данных (страховка ~минуты)
Убеждаемся, что данные распакованы в `<class>/*.avi`, PyAV декодит реальные
ролики, сплиты сходятся и train-цикл крутится на GPU. **Если выше `--check`
показал 0 clips у каких-то классов — остановитесь и пришлите вывод (поправим
фильтр), не запускайте полный прогон.**

In [ ]:
!python -m src.train --model tsn --epochs 1 --limit 40 --batch-size 4 --no-freeze
print("\nФАЗА 1 ок -> можно запускать полный прогон (ячейка ниже).")

## 8. ФАЗА 2 — полное обучение 5 моделей
Порядок **от надёжных к рискованным**: TSN → TSM → R(2+1)D → I3D → VideoMAE.
К моменту возможного OOM VideoMAE четыре модели уже обучены и сохранены.
Если VideoMAE падает — автоматически обучается запасная 5-я (**TimeSformer**),
так что в сравнении всегда ≥5 архитектур.

После каждой модели сразу считаются метрики (`evaluate`) → и чекпоинт, и
`results/` лежат на Drive. Если рантайм умрёт — перезапустите ячейки 1-6 и
эту: готовые модели не переобучаются с нуля (чекпоинты на Drive).

По умолчанию backbone заморожен (учится голова) — быстро и экономит память;
для полного дообучения добавьте `--no-freeze`.

In [ ]:
import subprocess, sys, os

PROJ = "/content/venya"
if not os.path.isdir(PROJ):
    raise SystemExit("/content/venya отсутствует — рантайм был сброшен. "
                     "Перезапустите ячейки 1-6 (Drive, clone, install, download, "
                     "sanity, symlink), затем эту.")
os.chdir(PROJ)

EPOCHS = 15
ORDER = ["tsn", "tsm", "r2plus1d", "i3d", "videomae"]

def run(cmd):
    # stream output line-by-line so it is VISIBLE in Colab (Colab captures
    # Python print, not a child's stdout) and failures are loud.
    print(">>", " ".join(cmd[2:]), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    return p.returncode

def done(model):
    return os.path.exists(f"results/{model}_metrics.json")

def train_then_eval(model, train_extra=None):
    rc = run([sys.executable, "-m", "src.train", "--model", model,
              "--epochs", str(EPOCHS)] + (train_extra or []))
    if rc != 0:
        print(f"!! train {model} завершился с кодом {rc}", flush=True)
        return False
    run([sys.executable, "-m", "src.evaluate", "--model", model])
    return True

for m in ORDER:
    print("=" * 70, m, flush=True)
    if done(m):
        print(f"уже готов (results/{m}_metrics.json на Drive) — пропуск", flush=True)
        continue
    ok = train_then_eval(m)
    if not ok and m == "videomae":
        print("VideoMAE упал (вероятно OOM) -> запасная 5-я: TimeSformer", flush=True)
        if not (done("timesformer") or train_then_eval("timesformer")):
            print("TimeSformer тоже упал -> пробуем VideoMAE в tiny-режиме", flush=True)
            train_then_eval("videomae", ["--tiny"])
print("\nФАЗА 2 завершена.", flush=True)

## 9. Сравнение и отчёт

In [ ]:
!python -m src.compare
!python -m reports.generate_report
print("comparison + report готовы (results/, reports/out/ на Drive)")

## 10. ФАЗА 3 — анализ (длина клипа + примеры ошибок)
Влияние числа кадров на качество (по моделям с переменным входом) и сбор
3 удачных + 3 ошибочных примеров лучшей модели (с приоритетом спорных пар).

In [ ]:
!cd /content/venya && python -m src.frame_sweep
!cd /content/venya && python -m src.find_examples --model videomae

## 11. Выгрузить результаты
Собирает results/ (метрики, confusion, графики, frame_sweep), отчёт и примеры
в один архив и скачивает — пришлите его, чтобы заполнить README/анализ.

In [ ]:
import os
os.makedirs("/content/bundle", exist_ok=True)
# cp -L разыменовывает симлинки results/ -> Drive
!cp -rL /content/venya/results /content/bundle/results
!cp -rL /content/venya/reports/out /content/bundle/report_out
!cp -rL /content/venya/reports/examples /content/bundle/examples 2>/dev/null || true
!cd /content/bundle && zip -qr /content/venya_results.zip .
from google.colab import files
files.download("/content/venya_results.zip")

Чекпоинты (.pt) и метрики также лежат на Drive в `MyDrive/venya/`.